## SQL START to FINISH

Entities & Attributes

|CreditCardCompany (2)|BankAccount (3)|Customer (3)|CreditCard (4)|
|--|--|--|--|
|company_id (PK) > C01, C04, C06|account_number (PK) > B01, B02, B05|customer_id|card_number (PK)|
|company_name (e.g., VISA, MASTERCARD)|customer_id > P01, P03, P54|customer_initials|company_id (FK → CreditCardCompany)|
||balance|DOB|bank_account (FK → BankAccount)|
||||EXPIRY|


In [ ]:
DROP TABLE IF EXISTS new_data.creditcardcompany

DROP SCHEMA IF EXISTS new_data

CREATE SCHEMA new_data

CREATE TABLE new_data.creditcardcompany
(
 company_id VARCHAR(3) NOT NULL
)

ALTER TABLE new_data.creditcardcompany
ADD comapny_name VARCHAR(20)

INSERT INTO new_data.creditcardcompany
VALUES
('C01','VISA'),
('C02','Master Card')

ALTER TABLE new_data.creditcardcompany
ADD CONSTRAINT PK_ccid PRIMARY KEY NONCLUSTERED (company_id) NOT ENFORCED 

CREATE TABLE new_data.bankaccount
(
 account_number VARCHAR(3) NOT NULL,
 customer_id VARCHAR(3),
 balance INT
)

ALTER TABLE new_data.bankaccount
ADD CONSTRAINT PK_cust_id PRIMARY KEY NONCLUSTERED (customer_id) NOT ENFORCED 

INSERT INTO new_data.bankaccount
VALUES
('B01','P01',150000),
('B02','P01',25000),
('B03','P02',350000),
('B04','P02',4000)

CREATE TABLE new_data.customer
(
 customer_id VARCHAR(3),
 DOB DATE,
 customer_initials VARCHAR(6)
)

INSERT INTO new_data.customer
VALUES
('P01','2000-01-04','LIJ'),
('P01','2002-04-05','DEF')

update new_data.customer
set customer_id = 'P02'
where customer_initials = 'DEF'

CREATE TABLE new_data.creditcard
(
 cardnumber INT NOT NULL,
 company_id VARCHAR(3),
 bankac VARCHAR(3),
 expiry DATE
)

INSERT INTO new_data.creditcard
VALUES
(839242, 'C01', 'B01', '2030-01-01'),
(738421, 'C02', 'B02', '2029-06-01'),
(213211, 'C01', 'B03', '2028-07-01')


In [ ]:
% skip
ALTER TABLE child_table
ADD CONSTRAINT constraint_name
FOREIGN KEY (child_column)
REFERENCES parent_table (parent_column)
NOT ENFORCED;
Example
Let’s say you have:

Parent table: Customers with CustomerID as the primary key
Child table: Orders with CustomerID referencing Customers.CustomerID

-- Add NOT ENFORCED foreign key
ALTER TABLE Orders
ADD CONSTRAINT FK_Orders_Customers
FOREIGN KEY (CustomerID)
REFERENCES Customers (CustomerID)
NOT ENFORCED;
Key Points
NOT ENFORCED:
No automatic validation of matching values.
Exists only in metadata for documentation and BI tools.
Order of creation:
The referenced column in the parent table must already exist and be defined as a primary key (even if NOT ENFORCED).
Data integrity:
You must manually ensure that all CustomerID values in Orders exist in Customers.
Pre-check Script (Recommended)
Before adding the foreign key:


-- Find orphaned rows in child table
SELECT o.CustomerID
FROM Orders o
LEFT JOIN Customers c ON o.CustomerID = c.CustomerID
WHERE c.CustomerID IS NULL;
If this returns 0 rows, you can safely add the foreign key.

If you want, I can prepare a Fabric-safe script that:

Checks for orphaned rows.
Creates the NOT ENFORCED foreign key only if the data is clean.
Do you want me to prepare that?

### CTE Common Table Expressions

In [ ]:
WITH BalanceCTE AS (
    SELECT 
        customer_id
    FROM new_data.bankaccount
    WHERE balance > 55000
)
SELECT *
FROM BalanceCTE;



WITH BalanceCTE AS (
    SELECT 
        *
    FROM new_data.bankaccount
    WHERE balance > 55000
),
customerCTE AS (
select *
from BalanceCTE
where customer_id != 'P02'
)
SELECT *
FROM customerCTE;

### Parameters

In [ ]:
DECLARE @given_date DATETIME = '2024-04-03';

select *,
lower(concat(cust_init, cust_init, CAST(cust_id AS VARCHAR(10)))) as new_name,
DATEDIFF(MONTH, DOB, CAST(GETDATE() AS DATE)) AS MonthDiff,
DATEDIFF(YEAR,   @given_date, GETDATE()) AS YearsDiffConst
from bank_db.baknac

### Views

In [ ]:
CREATE OR ALTER VIEW name_lower
AS
select *,
concat(customer_initials, lower(customer_initials)) as new_name
from new_data.customer


SELECT TOP (100) [customer_id],
			[DOB],
			[customer_initials],
			[new_name]
FROM [customer].[name_lower]


### Scalar Functions

In [ ]:
CREATE FUNCTION [new_data].[scalarValuedFunction1] (
	@param1 INT,
	@param2 INT
)
RETURNS INT AS BEGIN RETURN
	@param1 + @param2
END

-----------------------------------------------------

SELECT 
    *,
    new_data.scalarValuedFunction1(balance, 100) AS TotalAmount
FROM new_data.bankaccount

account_number	customer_id	balance	TotalAmount
B01	P01	150000	150100
B02	P01	25000	25100
B03	P02	350000	350100
B04	P02	4000	4100

### Table Valued Function

**Type 1**

In [ ]:
CREATE FUNCTION [new_data].[tableValuedFunction1] (
	@Parameter1 INT,
	@Parameter2 INT
)
RETURNS TABLE AS RETURN (
	SELECT @parameter1 + @parameter2
	AS RESULT
)

-----------------------------------------------------

SELECT *
FROM new_data.tableValuedFunction1(10, 20)

RESULT
30

**Type 2**

In [ ]:
CREATE FUNCTION new_data.tableValuedFunction2 (@Parameter1 INT,@Parameter2 FLOAT)
RETURNS TABLE
AS
RETURN (
    SELECT (@Parameter1 + @Parameter2) AS Result
);

-----------------------------------------------------

SELECT 
    b.account_number,
    b.customer_id,
    tvf.Result AS AmountInterest
FROM new_data.bankaccount b
CROSS APPLY new_data.tableValuedFunction2(b.balance, b.balance * 0.04) AS tvf;

account_number	customer_id	AmountInterest
B01	P01	156000
B02	P01	26000
B03	P02	364000
B04	P02	4160

**Type 3**

In [ ]:
CREATE FUNCTION new_data.calculate_interestedamount (@ROI FLOAT)
RETURNS TABLE
AS
RETURN
(
    SELECT 
        b.account_number,
        b.customer_id,
        b.balance * (1 + @ROI/100) AS AmountInterest
    FROM new_data.bankaccount b
);

-----------------------------------------------------

SELECT * 
FROM new_data.calculate_interestedamount(4);

account_number	customer_id	AmountInterest
B01	P01	156000
B02	P01	26000
B03	P02	364000
B04	P02	4160

**Type 4**

In [ ]:
CREATE FUNCTION new_data.GetCardsExpiringSoon(validmonths INT)
RETURNS TABLE
AS
RETURN
(
    SELECT 
        cardnumber,
        company_id,
        bankac,
        expiry
    FROM new_data.creditcard
    WHERE expiry < DATEADD(MONTH, validmonths, CAST(GETDATE() AS DATE))  -- Less than 2 months from today
      AND expiry >= CAST(GETDATE() AS DATE)                    -- Not already expired
);
GO

-----------------------------------------------------


SELECT *
FROM new_data.GetCardsExpiringSoon(30);

cardnumber	company_id	bankac	expiry
213211	C01	B03	2028-07-01

### Stored Procedures